# Stage 09 — Homework Starter Notebook

In the lecture, we learned how to create engineered features. Now it’s your turn to apply those ideas to your own project data.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install matplotlib

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Example synthetic data (replace with your project dataset)
np.random.seed(0)
n = 100
df = pd.DataFrame({
    'income': np.random.normal(60000, 15000, n).astype(int),
    'monthly_spend': np.random.normal(2000, 600, n).astype(int),
    'credit_score': np.random.normal(680, 50, n).astype(int),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'default_flag': np.random.choice([0, 1], n, p=[0.8, 0.2]),
})
df.head()

,income,monthly_spend,credit_score,region,default_flag
0,86460,3129,661,East,0
1,66002,1191,668,North,0
2,74681,1237,734,West,0
3,93613,2581,712,South,1
4,88013,1296,712,East,1


## Engineered features — one is worked below; add at least two more

At least one of yours must encode a categorical column. `region` is the categorical
column here; the lecture shows three ways to encode it (one-hot, label, frequency).

In [2]:
# Worked example 1 of 3 — keep it, or swap in columns of your own.
df['spend_income_ratio'] = df['monthly_spend'] / df['income']
# Write the rationale in the markdown cell below.

### Rationale for Feature 1


`spend_income_ratio` measures monthly spending relative to income. This feature captures proportional spending behavior rather than spending level alone.

This idea connects to Stage 08 EDA because `income` showed a meaningful relationship with `spend`, and using a ratio may reveal customers who spend unusually high or low amounts relative to their income.

A higher ratio may indicate greater spending pressure relative to available income, which could be useful for modeling financial behavior or default risk.

In [3]:
# TODO: Add another feature
# Example: df['rolling_spend_mean'] = df['monthly_spend'].rolling(3).mean()
df['income_credit_interaction'] = df['income'] * df['credit_score']


### Rationale for Feature 2
Explain why this feature may help a model. Reference your EDA.

### Rationale for Feature 2

`income_credit_interaction` combines `income` and `credit_score` to capture their joint effect.

This feature may help because income and credit quality together can describe a customer's financial profile more completely than either variable alone. A customer with both high income and a high credit score may have a different risk profile from someone who is high on only one of the two measures.

This follows the Stage 08 EDA idea of turning relationships between variables into testable features.

In [4]:
# TODO: Add a third feature. At least one of your three must ENCODE A CATEGORICAL
#   column - `region` is the one in this dataset.
#
#   The lecture shows three ways (section 'Categorical Encoding'):
#     one-hot     pd.get_dummies(df, columns=['region'])
#     label       LabelEncoder().fit_transform(df['region'])
#     frequency   df['region'].map(df['region'].value_counts(normalize=True))
#
#   Pick one and say WHY you picked it in the markdown cell below - the three are not
#   interchangeable, and that choice is the point of the exercise.
df['region_frequency'] = df['region'].map(
    df['region'].value_counts(normalize=True)
)

### Rationale for Feature 3
Explain why this feature may help a model. Reference your EDA. If this is your
categorical encoding, say why you chose that encoding over the other two.

### Rationale for Feature 3

I used frequency encoding for `region`, replacing each region with its relative frequency in the dataset.

I chose frequency encoding because the Stage 08 EDA showed that the four regions were reasonably balanced, so their relative frequencies provide a compact numeric representation without creating multiple dummy columns.

I did not use label encoding because assigning integers such as 0, 1, 2, and 3 could incorrectly imply an ordinal relationship between regions. One-hot encoding would also be valid, but it would expand one categorical column into several columns.

This feature allows the model to use information about how common each region is while keeping the feature space simple.

In [5]:
feature_cols = [
    'spend_income_ratio',
    'income_credit_interaction',
    'region_frequency'
]

corr_with_target = df[
    feature_cols + ['default_flag']
].corr(numeric_only=True)['default_flag'].drop('default_flag')

corr_with_target


spend_income_ratio          -0.036101
income_credit_interaction    0.176136
region_frequency             0.033070
Name: default_flag, dtype: float64

### Feature Correlation Check

The engineered features were compared with `default_flag` using simple correlations.

These correlations are used only as an initial screening tool. A weak correlation does not necessarily mean a feature is useless, because the relationship with the target may be nonlinear or may depend on interactions with other variables.
- `spend_income_ratio` has a correlation of about `-0.04` with `default_flag`, which is very weak. This suggests that the ratio may not have a strong linear relationship with default risk by itself.

- `income_credit_interaction` has the strongest correlation with `default_flag`, about `0.18`. This is still modest, but it suggests that combining income and credit score may capture more useful information than the other engineered features.

- `region_frequency` has a correlation of about `0.03` with `default_flag`, indicating almost no linear relationship in this synthetic dataset. However, categorical region effects may still be nonlinear or interact with other variables.

In [6]:
from src.features import (
    add_spend_income_ratio,
    add_income_credit_interaction,
    add_region_frequency
)

df_test = df.copy()
df_test = add_spend_income_ratio(df_test)
df_test = add_income_credit_interaction(df_test)
df_test = add_region_frequency(df_test)

df_test[
    [
        'spend_income_ratio',
        'income_credit_interaction',
        'region_frequency'
    ]
].head()

,spend_income_ratio,income_credit_interaction,region_frequency
0,0.036190,57150060,0.28
1,0.018045,44089336,0.22
2,0.016564,54815854,0.22
3,0.027571,66652456,0.28
4,0.014725,62665256,0.28
